In [ ]:
# ============================================================
# SECTION 1 — PROJECT SETUP & IMPORTS
# ============================================================
# Minimal dependency set. Each import is justified:
#   pandas  - CSV loading and tabular handling of 900 records
#   re      - ALL pattern matching (sensitive values, frames, dates, times)
#   json    - structured output files required by the assessment
#   pathlib - OS-independent relative paths (Windows local + Linux cloud)
#
# Deliberately NOT imported:
#   numpy / scikit-learn - no ML model is shipped (locked methodology)
#   dateutil             - its fuzzy parsing invents dates from vague text,
#                          which the assessment explicitly prohibits
# ============================================================

import re
import json
import sys
import platform
from pathlib import Path

import pandas as pd

print("Environment")
print(f"  Python        : {sys.version.split()[0]}")
print(f"  Platform      : {platform.system()} {platform.release()}")
print(f"  pandas        : {pd.__version__}")
print("\nImports loaded: re, json, sys, platform, pathlib.Path, pandas")

In [ ]:
# ------------------------------------------------------------
# Locate the project root robustly.
# Preferred: the notebook's own folder (relative paths, portable).
# Fallback : walk up parent folders looking for data/messages.csv.
# Override : set PROJECT_ROOT_OVERRIDE if Jupyter was launched elsewhere.
# ------------------------------------------------------------

PROJECT_ROOT_OVERRIDE = None   # e.g. r"C:\KaStack_AI_ML_Assessment"

def find_project_root(start: Path, marker: str = "data/messages.csv") -> Path:
    """Return the first folder at or above `start` that contains `marker`."""
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    return start   # nothing found; keep cwd so the check below reports MISSING

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MESSAGES_PATH      = DATA_DIR / "messages.csv"
MANDATORY_IDS_PATH = DATA_DIR / "mandatory_demo_ids.csv"

OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Working dir  : {Path.cwd()}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print()

# Presence + size only. No file CONTENT is read or printed here:
# the supplied dataset is private and must not appear in notebook
# output that may later be committed or screen-recorded.
all_found = True
for label, path in [("messages.csv", MESSAGES_PATH),
                    ("mandatory_demo_ids.csv", MANDATORY_IDS_PATH)]:
    if path.exists():
        print(f"  FOUND    {label:<24} ({path.stat().st_size:,} bytes)")
    else:
        print(f"  MISSING  {label:<24} -> expected at {path}")
        all_found = False

print()
print("Section 1 setup OK" if all_found else "Section 1 setup INCOMPLETE - fix paths above")

In [ ]:
# ============================================================
# SECTION 2 — CONFIGURATION
# ============================================================
# All decisions live here. Logic lives in later sections.
# ============================================================

# --- The six required categories --------------------------------------
# snake_case values match the JSON example in the assessment PDF:
#   {"message_id": "MSG_001", "category": "action_required", ...}

CATEGORY_SENSITIVE   = "sensitive_information"
CATEGORY_PROMOTIONAL = "promotional"
CATEGORY_MEETING     = "meeting_or_event"
CATEGORY_ACTION      = "action_required"
CATEGORY_PERSONAL    = "personal_information"
CATEGORY_GENERAL     = "general_information"

# --- Precedence ladder: first match wins ------------------------------
# Order is a deliberate safety and correctness decision:
#
# 1. SENSITIVE first  - a message carrying a live credential is sensitive
#                       whatever else it asks. "Use password X to sign in"
#                       reads as an instruction but is a credential
#                       disclosure, and mis-ranking it would leak a value.
# 2. PROMOTIONAL next - a discount-code frame is decisive and outranks the
#                       polite request wrapper marketing text often uses.
# 3. MEETING          - explicit scheduling frames (date + time + place).
# 4. ACTION           - obligation/request frames without scheduling.
# 5. PERSONAL         - self-disclosure that is not a credential.
# 6. GENERAL          - documented fallback, never a silent default.

CATEGORY_PRECEDENCE = [
    CATEGORY_SENSITIVE,
    CATEGORY_PROMOTIONAL,
    CATEGORY_MEETING,
    CATEGORY_ACTION,
    CATEGORY_PERSONAL,
    CATEGORY_GENERAL,
]

# Display labels for the notebook tables and the Streamlit UI.
CATEGORY_LABELS = {
    CATEGORY_SENSITIVE:   "Sensitive Information",
    CATEGORY_PROMOTIONAL: "Promotional",
    CATEGORY_MEETING:     "Meeting or Event",
    CATEGORY_ACTION:      "Action Required",
    CATEGORY_PERSONAL:    "Personal Information",
    CATEGORY_GENERAL:     "General Information",
}

assert len(CATEGORY_PRECEDENCE) == 6
assert set(CATEGORY_PRECEDENCE) == set(CATEGORY_LABELS)

print("Categories configured (precedence order):")
for rank, cat in enumerate(CATEGORY_PRECEDENCE, start=1):
    print(f"  {rank}. {CATEGORY_LABELS[cat]:<24} ({cat})")

In [ ]:
# --- Known noise prefixes ---------------------------------------------
# Conversational filler prepended to a core sentence. These carry no
# category signal and are stripped into a separate `core_text` column in
# Section 5. The original `message` column is never modified.
#
# "Can you help?" is the important one. It is NOT an action signal: it
# appears on promotional, general and personal messages alike. Treating
# it as an imperative is exactly the bad logic the brief prohibits.
#
# "Please note:" carries a colon. The sensitive template
# "Please note my bank account number ..." does not. That colon is the
# only thing separating a prefix from message content, so it must stay.
#
# Note the em dash (U+2014) in "Just checking—", with no trailing space.

NOISE_PREFIXES = [
    "For today:",
    "FYI:",
    "One more thing:",
    "Important:",
    "Just checking\u2014",
    "Please note:",
    "Quick update:",
    "Can you help?",
    "Hi,",
]

# Prefixes can stack ("Hi, FYI: ..."). Bounded so the strip loop always
# terminates, even on unexpected input.
MAX_PREFIX_STRIP_PASSES = 4

# --- Confidence bands --------------------------------------------------
# IMPORTANT: these are RULE-STRENGTH scores, not model probabilities.
# No classifier is trained, so nothing here is a statistical likelihood.
# Each band has a defined, documented meaning:

CONFIDENCE = {
    "exact_frame": 0.95,  # highly specific structural pattern matched
                          # (credential format, full scheduling frame)
    "strong":      0.90,  # clear frame + corroborating signal (explicit date)
    "moderate":    0.75,  # clear frame, no corroborating detail
    "hedged":      0.55,  # frame present but hedged: "could", "may",
                          # "might", "sometime" - intent real, details vague
    "fallback":    0.50,  # no frame matched; assigned the default category
}

CONFIDENCE_MEANING = {
    0.95: "Highly specific pattern - very low ambiguity",
    0.90: "Clear frame supported by a second explicit signal",
    0.75: "Clear frame, no corroborating detail",
    0.55: "Hedged or under-specified language - genuinely uncertain",
    0.50: "No frame matched - fallback category",
}

print(f"Noise prefixes configured : {len(NOISE_PREFIXES)}")
print(f"Max strip passes          : {MAX_PREFIX_STRIP_PASSES}")
print("\nConfidence bands (rule strength, NOT model probability):")
for name, score in CONFIDENCE.items():
    print(f"  {score:.2f}  {name:<12} {CONFIDENCE_MEANING[score]}")

In [ ]:
# --- Sensitivity policy ------------------------------------------------
# Maps each sensitivity type to a risk level and a recommended action.
# Recommended actions are restricted to the vocabulary given in the
# assessment PDF - no invented action names.

RISK_HIGH, RISK_MEDIUM, RISK_LOW = "high", "medium", "low"

ACTION_SAFE_LOCAL  = "safe_to_process_locally"
ACTION_CONFIRM     = "ask_for_confirmation"
ACTION_DO_NOT_STORE = "do_not_store"
ACTION_NO_EXTERNAL = "do_not_send_to_external_service"

# Rationale for the split:
#   do_not_store  - reusable secrets. Persisting them creates a standing
#                   liability even inside a local system.
#   no_external   - financial identifiers. Storage is not the main hazard;
#                   transmission to a third party is.
#   confirm       - personal data that is sensitive but often legitimately
#                   needed, so a human decides rather than a blanket block.

SENSITIVITY_POLICY = {
    "one_time_password": (RISK_HIGH,   ACTION_DO_NOT_STORE),
    "password":          (RISK_HIGH,   ACTION_DO_NOT_STORE),
    "auth_token":        (RISK_HIGH,   ACTION_DO_NOT_STORE),
    "recovery_code":     (RISK_HIGH,   ACTION_DO_NOT_STORE),
    "bank_account":      (RISK_HIGH,   ACTION_NO_EXTERNAL),
    "card_number":       (RISK_HIGH,   ACTION_NO_EXTERNAL),
    "id_number":         (RISK_HIGH,   ACTION_DO_NOT_STORE),
    "health_data":       (RISK_HIGH,   ACTION_CONFIRM),
    "home_address":      (RISK_MEDIUM, ACTION_CONFIRM),
    "phone_number":      (RISK_MEDIUM, ACTION_CONFIRM),
}

# Masking: fixed-width so the mask does not reveal the original length.
MASK_CHAR  = "*"
MASK_WIDTH = 6

# --- Extraction sentinels ----------------------------------------------
# Two DIFFERENT meanings, deliberately kept apart:
#   None        -> the message never referred to this slot at all
#   UNRESOLVED  -> the message referred to it but gave no explicit value
#                  ("sometime next week"). We refuse to guess.

UNRESOLVED = "unresolved"

PRIORITY_HIGH, PRIORITY_MEDIUM, PRIORITY_LOW = "high", "medium", "low"

print(f"Sensitivity types configured: {len(SENSITIVITY_POLICY)}")
for stype, (risk, action) in SENSITIVITY_POLICY.items():
    print(f"  {stype:<20} risk={risk:<7} action={action}")
print(f"\nMask format   : {MASK_CHAR * MASK_WIDTH}")
print(f"Sentinels     : None = slot absent | '{UNRESOLVED}' = referenced but not explicit")

In [ ]:
# ============================================================
# SECTION 3 — DATASET LOADING
# ============================================================
# Load only. All checking happens in Section 4.
# Nothing here prints message content: the dataset is private and
# this notebook will be screen-recorded.
# ============================================================

# dtype=str          - message_id "MSG_0001" and timestamps must not be
#                      coerced by pandas type inference
# keep_default_na=False - a blank cell stays "" instead of becoming NaN,
#                      so Section 4 can report it instead of it vanishing
# encoding="utf-8-sig" - harmless if no BOM; strips one if present

messages_df = pd.read_csv(
    MESSAGES_PATH,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

print("messages.csv loaded")
print(f"  Shape        : {messages_df.shape[0]} rows x {messages_df.shape[1]} columns")
print(f"  Columns      : {list(messages_df.columns)}")
print(f"  Dtypes       : {sorted(set(messages_df.dtypes.astype(str)))}")
print()

# Structure preview WITHOUT content: column names and value lengths only.
print("Column value lengths (characters):")
for col in messages_df.columns:
    lengths = messages_df[col].astype(str).str.len()
    print(f"  {col:<12} min={lengths.min():>3}  max={lengths.max():>3}  mean={lengths.mean():>6.1f}")

In [ ]:
# The mandatory-ID file ships with a UTF-8 BOM and CRLF line endings.
# Without encoding="utf-8-sig" the first value becomes "\ufeffMSG_0002",
# which looks correct when printed but fails every membership test.

mandatory_raw = pd.read_csv(
    MANDATORY_IDS_PATH,
    dtype=str,
    encoding="utf-8-sig",
)

# Tolerate a differently-named first column rather than assuming a header.
id_column = "message_id" if "message_id" in mandatory_raw.columns else mandatory_raw.columns[0]

MANDATORY_IDS = [str(v).strip() for v in mandatory_raw[id_column] if str(v).strip()]

print("mandatory_demo_ids.csv loaded")
print(f"  Column used   : {id_column!r}")
print(f"  IDs found     : {len(MANDATORY_IDS)}")
print(f"  Unique IDs    : {len(set(MANDATORY_IDS))}")
print(f"  BOM remaining : {any(chr(0xFEFF) in i for i in MANDATORY_IDS)}")
print()
print("  IDs (these are identifiers, not message content - safe to display):")
print(f"    {MANDATORY_IDS}")

In [ ]:
# Sender names are metadata, not message content, so they are safe to show.
# This is also the evidence for a design decision made in Section 7:
# sender alone must NOT drive classification.

sender_counts = messages_df["sender"].value_counts()

print(f"Distinct senders: {sender_counts.shape[0]}\n")
print(f"{'sender':<20} {'messages':>9}")
print("-" * 30)
for sender, count in sender_counts.items():
    print(f"{sender:<20} {count:>9}")

print(f"\nTotal: {sender_counts.sum()} messages")

In [ ]:
# ============================================================
# SECTION 4 — DATASET VALIDATION
# ============================================================
# Report, never repair. Several findings below are real properties
# of the supplied data that the assessment expects us to preserve.
# ============================================================

VALIDATION_ERRORS   = []   # make processing unsafe or impossible
VALIDATION_WARNINGS = []   # notable, but processing continues
VALIDATION_STATS    = {}   # facts recorded for the README and the video

def check_error(condition_ok: bool, message: str) -> None:
    """Record a fatal problem when condition_ok is False."""
    if not condition_ok:
        VALIDATION_ERRORS.append(message)

def check_warn(condition_ok: bool, message: str) -> None:
    """Record a non-fatal observation when condition_ok is False."""
    if not condition_ok:
        VALIDATION_WARNINGS.append(message)

print("Validation scaffold ready.")
print("  check_error() -> blocks processing")
print("  check_warn()  -> records an observation, processing continues")

In [ ]:
# --- Expected columns --------------------------------------------------
REQUIRED_COLUMNS = ["message_id", "timestamp", "sender", "message"]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in messages_df.columns]
extra_cols   = [c for c in messages_df.columns if c not in REQUIRED_COLUMNS]

check_error(not missing_cols, f"Missing required columns: {missing_cols}")
check_warn(not extra_cols,    f"Unexpected extra columns: {extra_cols}")

# --- Row count ---------------------------------------------------------
EXPECTED_ROW_COUNT = 900
row_count = len(messages_df)
VALIDATION_STATS["row_count"] = row_count
check_warn(row_count == EXPECTED_ROW_COUNT,
           f"Row count {row_count} != expected {EXPECTED_ROW_COUNT}")

# --- Missing / blank values -------------------------------------------
blank_counts = {c: int((messages_df[c].astype(str).str.strip() == "").sum())
                for c in REQUIRED_COLUMNS}
total_blanks = sum(blank_counts.values())
VALIDATION_STATS["blank_cells"] = total_blanks
check_warn(total_blanks == 0, f"Blank cells present: {blank_counts}")

# --- Duplicates --------------------------------------------------------
dup_rows = int(messages_df.duplicated().sum())
dup_ids  = int(messages_df["message_id"].duplicated().sum())
dup_text = int(messages_df["message"].duplicated().sum())

VALIDATION_STATS["duplicate_rows"]     = dup_rows
VALIDATION_STATS["duplicate_ids"]      = dup_ids
VALIDATION_STATS["duplicate_messages"] = dup_text

# Duplicate IDs are fatal: every output file is keyed by message_id.
check_error(dup_ids == 0, f"{dup_ids} duplicate message_id value(s)")
check_warn(dup_rows == 0, f"{dup_rows} fully duplicated row(s)")
check_warn(dup_text == 0, f"{dup_text} duplicate message text value(s)")

print(f"Columns          : {len(messages_df.columns)} present, {len(missing_cols)} missing")
print(f"Rows             : {row_count}")
print(f"Blank cells      : {total_blanks}")
print(f"Duplicate rows   : {dup_rows}")
print(f"Duplicate IDs    : {dup_ids}")
print(f"Duplicate texts  : {dup_text}")
print(f"Unique IDs       : {messages_df['message_id'].nunique()}")

In [ ]:
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"

# format= (not inference): a malformed timestamp becomes NaT and is
# reported, rather than being silently guessed into a plausible date.
parsed_ts = pd.to_datetime(
    messages_df["timestamp"], format=TIMESTAMP_FORMAT, errors="coerce"
)

unparseable = int(parsed_ts.isna().sum())
check_error(unparseable == 0,
            f"{unparseable} timestamp(s) do not match {TIMESTAMP_FORMAT}")

if unparseable == 0:
    is_chronological = bool(parsed_ts.is_monotonic_increasing)
    VALIDATION_STATS["timestamp_min"]     = str(parsed_ts.min())
    VALIDATION_STATS["timestamp_max"]     = str(parsed_ts.max())
    VALIDATION_STATS["already_chronological"] = is_chronological
    VALIDATION_STATS["duplicate_timestamps"]  = int(parsed_ts.duplicated().sum())

    # Reported here, acted on in Section 5. We do not reorder inside a
    # validation step.
    check_warn(is_chronological,
               "Rows are NOT chronologically ordered; Section 5 will sort by timestamp")

    print(f"Unparseable timestamps : {unparseable}")
    print(f"Range                  : {parsed_ts.min()}  ->  {parsed_ts.max()}")
    print(f"Span                   : {(parsed_ts.max() - parsed_ts.min()).days} days")
    print(f"Already chronological  : {is_chronological}")
    print(f"Duplicate timestamps   : {VALIDATION_STATS['duplicate_timestamps']}")
else:
    print(f"Unparseable timestamps : {unparseable}  <-- FATAL, see report")

In [ ]:
# --- Mandatory demonstration IDs --------------------------------------
dataset_ids  = set(messages_df["message_id"])
missing_ids  = [i for i in MANDATORY_IDS if i not in dataset_ids]

VALIDATION_STATS["mandatory_ids_expected"] = len(MANDATORY_IDS)
VALIDATION_STATS["mandatory_ids_found"]    = len(MANDATORY_IDS) - len(missing_ids)

check_error(len(MANDATORY_IDS) == 15,
            f"Expected 15 mandatory IDs, found {len(MANDATORY_IDS)}")
check_error(not missing_ids,
            f"Mandatory IDs absent from messages.csv: {missing_ids}")

print(f"Mandatory IDs expected : {len(MANDATORY_IDS)}")
print(f"Present in dataset     : {len(MANDATORY_IDS) - len(missing_ids)}")
print(f"Missing                : {missing_ids if missing_ids else 'none'}")

# --- Final report ------------------------------------------------------
print("\n" + "=" * 58)
print("DATASET VALIDATION REPORT")
print("=" * 58)

for key, value in VALIDATION_STATS.items():
    print(f"  {key:<26} : {value}")

print(f"\n  Errors   : {len(VALIDATION_ERRORS)}")
for e in VALIDATION_ERRORS:
    print(f"    ERROR   {e}")

print(f"  Warnings : {len(VALIDATION_WARNINGS)}")
for w in VALIDATION_WARNINGS:
    print(f"    WARNING {w}")

VALIDATION_PASSED = len(VALIDATION_ERRORS) == 0
print("\n  RESULT   :", "PASS - safe to process" if VALIDATION_PASSED else "FAIL - fix errors first")
print("=" * 58)

In [ ]:
# ============================================================
# SECTION 5 — MINIMAL TEXT PREPROCESSING
# ============================================================
# Only preprocessing justified by the actual data. The original
# `message` column is never modified.
# ============================================================

def strip_noise_prefixes(text: str):
    """Remove known conversational prefixes from the front of a message.

    Input : raw message text
    Output: (core_text, list_of_prefixes_removed)

    Prefixes can stack ("Hi, FYI: ..."), so this loops, bounded by
    MAX_PREFIX_STRIP_PASSES to guarantee termination.

    Two safety properties:
      1. Matching is anchored to the START of the string only. A prefix
         phrase occurring mid-sentence is content and is left alone.
      2. If stripping would consume the entire message, the original is
         returned instead. An empty core_text would fall through to
         General Information with no evidence behind it.

    Limitation: this is a closed list derived from the supplied dataset.
    An unseen prefix is left in place - the safe failure mode, since the
    frame matchers still find the core sentence behind it.
    """
    removed = []
    current = text.strip()

    for _ in range(MAX_PREFIX_STRIP_PASSES):
        matched = False
        for prefix in NOISE_PREFIXES:
            if current.startswith(prefix):
                current = current[len(prefix):].strip()
                removed.append(prefix)
                matched = True
                break
        if not matched:
            break

    if not current:
        return text.strip(), []

    return current, removed


def build_match_text(core_text: str) -> str:
    """Lowercased, whitespace-collapsed copy used ONLY for frame matching.

    Never displayed, never written to any output file. Sensitive detection
    does NOT use this - it needs the original casing.
    """
    return re.sub(r"\s+", " ", core_text).strip().lower()


# Quick behavioural checks on synthetic strings (no dataset content).
_checks = [
    ("Can you help? Buy one get one free. Use code SAVE10.", "Can you help?"),
    ("Just checking\u2014Tomorrow is a public holiday.",      "Just checking\u2014"),
    ("Hi, FYI: The portal is updated.",                       "Hi,"),
    ("Please note: Please note my bank account number 12345.", "Please note:"),
    ("Heads up: the lift is out.",                            None),
]

print("Prefix-stripping behaviour checks:")
for sample, expected_first in _checks:
    core, removed = strip_noise_prefixes(sample)
    first = removed[0] if removed else None
    status = "OK " if first == expected_first else "BAD"
    print(f"  {status} removed={removed} -> {core!r}")

In [ ]:
# Work on a copy. messages_df stays pristine as the loaded reference.
work_df = messages_df.copy()

# Original row position, kept so we can PROVE nothing was reordered.
work_df["source_order"] = range(len(work_df))

work_df["parsed_ts"] = pd.to_datetime(
    work_df["timestamp"], format=TIMESTAMP_FORMAT, errors="coerce"
)

_stripped = work_df["message"].map(strip_noise_prefixes)
work_df["core_text"]        = [s[0] for s in _stripped]
work_df["prefixes_removed"] = [s[1] for s in _stripped]
work_df["match_text"]       = work_df["core_text"].map(build_match_text)

# Chronological guarantee. The data is already sorted (verified in 4.3),
# so this is a stable no-op here. The message_id tie-break makes the order
# fully deterministic on any dataset, including the synthetic demo.
work_df = work_df.sort_values(["parsed_ts", "message_id"], kind="stable").reset_index(drop=True)

print(f"Rows after preprocessing : {len(work_df)}")
print(f"Columns added            : source_order, parsed_ts, core_text, prefixes_removed, match_text")
print(f"Chronological            : {work_df['parsed_ts'].is_monotonic_increasing}")
print(f"Reordered from source    : {not work_df['source_order'].is_monotonic_increasing}")
print(f"Original message column preserved : {work_df['message'].equals(messages_df['message'])}")

In [ ]:
from collections import Counter

prefix_counts = Counter(tuple(p) for p in work_df["prefixes_removed"])

print("Noise prefixes removed (count of messages):")
for combo, n in sorted(prefix_counts.items(), key=lambda kv: -kv[1]):
    label = " + ".join(combo) if combo else "(no prefix)"
    print(f"  {n:>4}  {label}")

n_stripped = int((work_df["prefixes_removed"].map(len) > 0).sum())
print(f"\nMessages with a prefix removed : {n_stripped}")
print(f"Messages left unchanged        : {len(work_df) - n_stripped}")
print(f"Empty core_text (must be 0)    : {int((work_df['core_text'].str.strip() == '').sum())}")

# --- Evidence for the 'Can you help? is not an action signal' claim ----
trap = work_df[work_df["prefixes_removed"].map(lambda x: "Can you help?" in x)]
promo_like = int(trap["match_text"].str.contains("use code").sum())

print(f"\n'Can you help?' prefixed messages : {len(trap)}")
print(f"  ...of which carry a discount code: {promo_like}")
print("  -> the prefix appears on non-action content, so it cannot imply Action Required")

In [ ]:
# ============================================================
# SECTION 6 — SENSITIVE INFORMATION DETECTION & MASKING
# ============================================================
# Patterns run against the ORIGINAL message text. match_text is
# lowercased, which would destroy ID-1234-XY, RC-88-KL, SAVE29 and
# tok_demo_... - all of which the patterns rely on.
#
# Each pattern captures ONLY the sensitive value in group 1. The
# surrounding words stay visible so the masked output remains
# readable and the detection stays auditable.
#
# Ordered most-specific first: an OTP is a digit run, but so is a
# phone number, so the labelled context word decides.
# ============================================================

SENSITIVE_PATTERNS = [
    # --- Credentials and secrets --------------------------------------
    ("one_time_password",
     re.compile(r"\bOTP\s+(?:is|:)\s*([A-Za-z0-9]{4,10})\b", re.IGNORECASE)),

    ("password",
     re.compile(r"\bpassword\s+(?:is|:)?\s*([A-Za-z0-9@#$%&*!_\-]{6,30})\b", re.IGNORECASE)),

    ("auth_token",
     re.compile(r"\b(?:access\s+)?token\s+(?:is|:)?\s*([A-Za-z0-9_\-]{8,60})\b", re.IGNORECASE)),

    ("recovery_code",
     re.compile(r"\brecovery\s+code\s+(?:is|:)?\s*([A-Za-z0-9\-]{4,30})\b", re.IGNORECASE)),

    # --- Financial identifiers ----------------------------------------
    ("card_number",
     re.compile(r"\bcard\s+number\s+(?:is|:)?\s*((?:\d[ \-]?){12,19})", re.IGNORECASE)),

    ("bank_account",
     re.compile(r"\bbank\s+account\s+number\s+(?:is|:)?\s*(\d{8,18})\b", re.IGNORECASE)),

    # --- Personal identifiers -----------------------------------------
    ("id_number",
     re.compile(r"\bidentification\s+number\s+(?:is|:)?\s*([A-Za-z0-9\-]{4,20})\b", re.IGNORECASE)),

    ("home_address",
     re.compile(r"\bhome\s+address\s+(?:is|:)?\s*([^.]{5,80})", re.IGNORECASE)),

    ("phone_number",
     re.compile(r"\bcontact\s+me\s+on\s+((?:\+?\d[\d\s\-]{7,15}\d))", re.IGNORECASE)),

    # --- Health data ---------------------------------------------------
    ("health_data",
     re.compile(r"\btest\s+result\s+(?:says|is|:)\s*([^.]{3,60})", re.IGNORECASE)),
]

# A REFERENCE to sensitive information that contains no actual value.
# These must NOT be classified Sensitive. Tracked separately so the
# distinction is visible rather than silently dropped.
REFERENCE_ONLY_PATTERNS = [
    re.compile(r"\b(?:login|account)\s+details\b", re.IGNORECASE),
    re.compile(r"\bcredentials\b", re.IGNORECASE),
]

print(f"Sensitive value patterns : {len(SENSITIVE_PATTERNS)}")
for stype, _ in SENSITIVE_PATTERNS:
    risk, action = SENSITIVITY_POLICY[stype]
    print(f"  {stype:<20} risk={risk:<7} action={action}")

print(f"\nReference-only patterns  : {len(REFERENCE_ONLY_PATTERNS)}")
print("  (mention a credential concept but carry no value -> NOT sensitive)")

# Every pattern type must have a policy entry, and vice versa.
_pattern_types = {s for s, _ in SENSITIVE_PATTERNS}
assert _pattern_types == set(SENSITIVITY_POLICY), (
    f"Policy/pattern mismatch: {_pattern_types ^ set(SENSITIVITY_POLICY)}"
)
print("\nPolicy coverage check: PASS - every pattern has a risk/action policy")

In [ ]:
def detect_sensitive(message: str) -> dict:
    """Detect sensitive VALUES in a message and produce a masked version.

    Input : original (unmodified) message text
    Output: dict with
              is_sensitive     bool  - an actual value was found
              is_reference     bool  - concept mentioned, no value present
              types            list  - sensitivity types detected
              risk             str   - highest risk across detections
              recommended_action str - action for the highest-risk type
              masked_text      str   - message with values replaced
              spans            list  - (type, start, end) for auditing
              reason           str   - what was matched, for explainability

    The raw captured value is used only to compute character offsets and
    is never returned or stored.
    """
    detections = []
    spans = []

    for stype, pattern in SENSITIVE_PATTERNS:
        for m in pattern.finditer(message):
            if m.group(1) and m.group(1).strip():
                detections.append(stype)
                spans.append((stype, m.start(1), m.end(1)))

    # Reference-only: concept named, but no value captured anywhere.
    is_reference = (not detections) and any(
        p.search(message) for p in REFERENCE_ONLY_PATTERNS
    )

    if not detections:
        return {
            "is_sensitive": False,
            "is_reference": is_reference,
            "types": [],
            "risk": None,
            "recommended_action": ACTION_SAFE_LOCAL,
            "masked_text": message,
            "spans": [],
            "reason": ("References credential information but contains no actual value"
                       if is_reference else "No sensitive value pattern matched"),
        }

    # Mask right-to-left so earlier offsets stay valid as we splice.
    masked = message
    for _stype, start, end in sorted(spans, key=lambda s: s[1], reverse=True):
        masked = masked[:start] + (MASK_CHAR * MASK_WIDTH) + masked[end:]

    # Highest risk wins when several types co-occur.
    risk_rank = {RISK_LOW: 0, RISK_MEDIUM: 1, RISK_HIGH: 2}
    top_type = max(detections, key=lambda t: risk_rank[SENSITIVITY_POLICY[t][0]])
    top_risk, top_action = SENSITIVITY_POLICY[top_type]

    unique_types = sorted(set(detections))
    return {
        "is_sensitive": True,
        "is_reference": False,
        "types": unique_types,
        "risk": top_risk,
        "recommended_action": top_action,
        "masked_text": masked,
        "spans": spans,
        "reason": (f"Detected {', '.join(unique_types)} value pattern"
                   f"{'s' if len(unique_types) > 1 else ''} in the message text"),
    }


# --- Behaviour checks on SYNTHETIC strings only ------------------------
# These are invented examples, not dataset content, so they are safe to
# display and safe to commit.
_samples = [
    "Your OTP is 483920. It expires in 10 minutes.",
    "My card number is 4111 1111 1111 1111.",
    "I will send the login details separately.",
    "The cafeteria closes at 8 PM.",
    "My home address is 12 Lake View Road, Chennai.",
]

print("Detection behaviour checks (synthetic examples):\n")
for s in _samples:
    r = detect_sensitive(s)
    flag = "SENSITIVE" if r["is_sensitive"] else ("REFERENCE" if r["is_reference"] else "clean    ")
    print(f"  {flag}  types={r['types']} risk={r['risk']}")
    print(f"             masked: {r['masked_text']}")
    print(f"             reason: {r['reason']}\n")

In [ ]:
# Detection runs on the ORIGINAL message column.
_results = work_df["message"].map(detect_sensitive)

work_df["is_sensitive"]        = [r["is_sensitive"] for r in _results]
work_df["is_reference_only"]   = [r["is_reference"] for r in _results]
work_df["sensitivity_types"]   = [r["types"] for r in _results]
work_df["risk_level"]          = [r["risk"] for r in _results]
work_df["recommended_action"]  = [r["recommended_action"] for r in _results]
work_df["masked_text"]         = [r["masked_text"] for r in _results]
work_df["sensitive_reason"]    = [r["reason"] for r in _results]

n_sens = int(work_df["is_sensitive"].sum())
n_ref  = int(work_df["is_reference_only"].sum())

print(f"Messages scanned            : {len(work_df)}")
print(f"Sensitive (value present)   : {n_sens}")
print(f"Reference only (no value)   : {n_ref}")
print(f"Clean                       : {len(work_df) - n_sens - n_ref}")

print("\nBreakdown by sensitivity type:")
type_counts = Counter(t for types in work_df["sensitivity_types"] for t in types)
for stype, n in sorted(type_counts.items(), key=lambda kv: -kv[1]):
    risk, action = SENSITIVITY_POLICY[stype]
    print(f"  {n:>4}  {stype:<20} risk={risk:<7} action={action}")

print("\nBy risk level:")
for risk, n in work_df.loc[work_df['is_sensitive'], 'risk_level'].value_counts().items():
    print(f"  {n:>4}  {risk}")

# Messages flagged sensitive but where masking changed nothing would be a
# silent leak. This must be zero.
_unmasked = int((work_df["is_sensitive"] & (work_df["masked_text"] == work_df["message"])).sum())
print(f"\nSensitive rows where masking changed nothing (must be 0): {_unmasked}")

In [ ]:
# ============================================================
# SECTION 7 - MESSAGE CLASSIFICATION
# ============================================================
# Structural frames, not keywords. Each rule is
#   (compiled_pattern, confidence, reason)
# The reason is emitted by the SAME rule that decides the
# category, so an explanation can never drift from the logic.
# ============================================================

def R(pattern):
    """Compile a case-insensitive frame pattern."""
    return re.compile(pattern, re.IGNORECASE)


# --- Promotional -------------------------------------------
PROMO_RULES = [
    (R(r"\buse code\s+[a-z0-9]+"),
     CONFIDENCE["exact_frame"],
     "marketing offer with a discount code"),
    (R(r"\byou may like our\b"),
     CONFIDENCE["moderate"],
     "product promotion without a discount code"),
]

# --- Meeting or event: explicit scheduling -----------------
MEETING_RULES = [
    (R(r"^are you available for .+ at .+ on \d{4}-\d{2}-\d{2}\?"),
     CONFIDENCE["exact_frame"],
     "availability request for a dated, timed event"),
    (R(r"^calendar update:"),
     CONFIDENCE["exact_frame"],
     "calendar entry with date, time and venue"),
    (R(r"^please join .+ on \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["exact_frame"],
     "invitation to a dated session"),
    (R(r"^reminder:.+happens on \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["exact_frame"],
     "reminder of a scheduled occurrence"),
    (R(r"\bis scheduled for \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["exact_frame"],
     "explicit scheduling statement"),
]

# --- Meeting or event: hedged, no fixed date ---------------
MEETING_VAGUE_RULES = [
    (R(r"^let us meet\b"),
     CONFIDENCE["hedged"],
     "meeting proposed without a fixed date"),
    (R(r"\b(?:review|meeting|session|call)\b.*\bcould be\b"),
     CONFIDENCE["hedged"],
     "possible session with hedged timing"),
]

print(f"promotional rules    : {len(PROMO_RULES)}")
print(f"meeting rules        : {len(MEETING_RULES)}")
print(f"meeting vague rules  : {len(MEETING_VAGUE_RULES)}")

In [ ]:
# --- Action required: explicit deadline present ------------
ACTION_RULES = [
    (R(r"^can you .+ before \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["strong"],
     "request frame with an explicit deadline"),
    (R(r"^don'?t forget to .+deadline is \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["strong"],
     "obligation reminder with a stated deadline"),
    (R(r"^i need you to .+ by \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["strong"],
     "direct assignment with a due date"),
    (R(r"^please (?:submit|complete|confirm|reply|send|upload) "
       r".+ by \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["strong"],
     "imperative request with a due date"),
    (R(r"\bis due on \d{4}-\d{2}-\d{2}"),
     CONFIDENCE["strong"],
     "task stated with an explicit due date"),
]

# --- Action required: imperative, no deadline --------------
ACTION_SOFT_RULES = [
    (R(r"^please (?:call|contact|check)\b"),
     CONFIDENCE["moderate"],
     "direct imperative request without a deadline"),
]

# --- Action required: hedged obligation --------------------
ACTION_HEDGED_RULES = [
    (R(r"^if possible,"),
     CONFIDENCE["hedged"],
     "conditional request without a deadline"),
    (R(r"^could you send it soon"),
     CONFIDENCE["hedged"],
     "vague request with an unspecified referent"),
    (R(r"\bmay be needed\b"),
     CONFIDENCE["hedged"],
     "possible obligation stated tentatively"),
]

# --- Personal information ----------------------------------
# NOTE: "(?:i|my)" is required. An earlier draft matched only
# "i" and silently dropped 5 messages beginning
# "Just so you know, my ..." into General Information.
PERSONAL_RULES = [
    (R(r"^for my profile,"),
     CONFIDENCE["exact_frame"],
     "first-person profile disclosure"),
    (R(r"^personal note:"),
     CONFIDENCE["exact_frame"],
     "explicitly labelled personal note"),
    (R(r"^just so you know, (?:i|my)\b"),
     CONFIDENCE["exact_frame"],
     "first-person preference disclosure"),
    (R(r"^remember that (?:i|my)\b"),
     CONFIDENCE["exact_frame"],
     "first-person standing preference"),
]

PERSONAL_HEDGED_RULES = [
    (R(r"^i might prefer\b"),
     CONFIDENCE["hedged"],
     "hedged personal preference"),
]

print(f"action rules         : {len(ACTION_RULES)}")
print(f"action soft rules    : {len(ACTION_SOFT_RULES)}")
print(f"action hedged rules  : {len(ACTION_HEDGED_RULES)}")
print(f"personal rules       : {len(PERSONAL_RULES)}")
print(f"personal hedged      : {len(PERSONAL_HEDGED_RULES)}")

In [ ]:
# Ladder order encodes two separate decisions:
#
# 1. CATEGORY precedence (Section 2): sensitive > promotional >
#    meeting > action > personal > general.
# 2. EVIDENCE strength within it: explicit-date meeting and
#    action frames are tested BEFORE hedged meeting frames, so a
#    fully specified message never loses to a vague pattern.

CLASSIFICATION_LADDER = [
    (CATEGORY_PROMOTIONAL, PROMO_RULES),
    (CATEGORY_MEETING,     MEETING_RULES),
    (CATEGORY_ACTION,      ACTION_RULES),
    (CATEGORY_MEETING,     MEETING_VAGUE_RULES),
    (CATEGORY_PERSONAL,    PERSONAL_RULES),
    (CATEGORY_ACTION,      ACTION_SOFT_RULES),
    (CATEGORY_ACTION,      ACTION_HEDGED_RULES),
    (CATEGORY_PERSONAL,    PERSONAL_HEDGED_RULES),
]


def classify_message(core_text: str, is_sensitive: bool) -> dict:
    """Assign one of the six categories to a single message.

    Input : core_text (prefixes stripped), is_sensitive flag
            computed in Section 6 from the ORIGINAL message.
    Output: dict with category, confidence, reason, matched_rule.

    Sensitive is checked first and outside the ladder: a message
    carrying a live credential is Sensitive regardless of what
    else it asks. Everything else falls through the ladder in
    order, first match wins.

    Limitation: frames are derived from the observed corpus. An
    unseen phrasing falls to General Information at 0.50 rather
    than being forced into a category - the honest failure mode.
    """
    if is_sensitive:
        return {
            "category": CATEGORY_SENSITIVE,
            "confidence": CONFIDENCE["exact_frame"],
            "reason": "contains an actual sensitive value",
            "matched_rule": "sensitive_value_detected",
        }

    for category, rules in CLASSIFICATION_LADDER:
        for pattern, confidence, reason in rules:
            if pattern.search(core_text):
                return {
                    "category": category,
                    "confidence": confidence,
                    "reason": reason,
                    "matched_rule": pattern.pattern[:48],
                }

    return {
        "category": CATEGORY_GENERAL,
        "confidence": CONFIDENCE["fallback"],
        "reason": ("no request, scheduling, promotional or "
                   "disclosure frame matched"),
        "matched_rule": "fallback",
    }


# --- Trap checks on synthetic strings ----------------------
_traps = [
    ("Buy one course and get one free. Use code SAVE10.",
     CATEGORY_PROMOTIONAL, "prefix trap: must not be action"),
    ("I might prefer evening meetings now.",
     CATEGORY_PERSONAL, "'meetings' must not trigger Meeting"),
    ("Let us meet sometime next week.",
     CATEGORY_MEETING, "vague meeting, low confidence"),
    ("The cafeteria closes at 8 PM.",
     CATEGORY_GENERAL, "plain fact -> fallback"),
]

print("Trap checks:\n")
for text, expected, note in _traps:
    r = classify_message(text, is_sensitive=False)
    ok = "OK " if r["category"] == expected else "BAD"
    print(f"  {ok} {r['category']:<22} conf={r['confidence']:.2f}")
    print(f"      {note}")
    print(f"      reason: {r['reason']}\n")

In [ ]:
_cls = [
    classify_message(core, sens)
    for core, sens in zip(work_df["core_text"], work_df["is_sensitive"])
]

work_df["category"]     = [c["category"] for c in _cls]
work_df["confidence"]   = [c["confidence"] for c in _cls]
work_df["reason"]       = [c["reason"] for c in _cls]
work_df["matched_rule"] = [c["matched_rule"] for c in _cls]

print(f"Messages classified : {len(work_df)}")
print(f"Missing category    : {int(work_df['category'].isna().sum())}")
print(f"Missing reason      : {int((work_df['reason'] == '').sum())}")
print(f"Invalid categories  : "
      f"{sorted(set(work_df['category']) - set(CATEGORY_PRECEDENCE))}")

print("\nCategory distribution:")
for cat in CATEGORY_PRECEDENCE:
    n = int((work_df["category"] == cat).sum())
    print(f"  {n:>4}  {CATEGORY_LABELS[cat]}")

print("\nConfidence distribution:")
for conf, n in work_df["confidence"].value_counts().sort_index().items():
    print(f"  {n:>4}  {conf:.2f}  {CONFIDENCE_MEANING[conf]}")

print("\nAll six categories populated:",
      set(work_df["category"]) == set(CATEGORY_PRECEDENCE))

In [ ]:
# ============================================================
# SECTION 8 - TASK / EVENT EXTRACTION
# ============================================================
# Named capture groups mean every extracted slot is a real span
# of the message. Nothing is inferred.
#
# Three DISTINCT states for a slot:
#   value        - explicitly present in the text
#   UNRESOLVED   - referenced, but no explicit value given
#                  ("sometime next week")
#   None         - never referenced at all
# ============================================================

TASK_FRAMES = [
    ("structured", R(r"^can you (?P<title>.+?) before "
                     r"(?P<date>\d{4}-\d{2}-\d{2})")),
    ("structured", R(r"^don'?t forget to (?P<title>.+?); "
                     r"deadline is (?P<date>\d{4}-\d{2}-\d{2})")),
    ("structured", R(r"^i need you to (?P<title>.+?) by "
                     r"(?P<date>\d{4}-\d{2}-\d{2})")),
    ("structured", R(r"^please (?P<title>(?:submit|complete|confirm"
                     r"|reply|send|upload).+?) by "
                     r"(?P<date>\d{4}-\d{2}-\d{2})")),
    ("structured", R(r"^(?P<title>.+?) is due on "
                     r"(?P<date>\d{4}-\d{2}-\d{2})")),

    # Imperative with a named person but NO temporal reference.
    ("partial",    R(r"^please call (?P<person>[A-Za-z]+) "
                     r"when you are free")),
    ("partial",    R(r"^if possible, (?P<title>.+?) "
                     r"(?P<hint>before the meeting)")),

    # Temporal reference present but not resolvable to a date.
    ("vague",      R(r"^could you send (?P<title>it) (?P<hint>soon)")),
    ("vague",      R(r"^the (?P<title>report) may be needed "
                     r"(?P<hint>tomorrow)")),
]

print(f"task frames: {len(TASK_FRAMES)}")
for strength, p in TASK_FRAMES:
    print(f"  {strength:<11} {p.pattern[:44]}")

In [ ]:
EVENT_FRAMES = [
    ("structured", R(r"^are you available for (?P<title>.+?) at "
                     r"(?P<time>\d{1,2}:\d{2}) on "
                     r"(?P<date>\d{4}-\d{2}-\d{2})\? "
                     r"Location: (?P<location>[^.]+)")),
    ("structured", R(r"^calendar update: (?P<title>[^,]+), "
                     r"(?P<date>\d{4}-\d{2}-\d{2}) at "
                     r"(?P<time>\d{1,2}:\d{2}), (?P<location>[^.]+)")),
    ("structured", R(r"^please join (?P<title>.+?) on "
                     r"(?P<date>\d{4}-\d{2}-\d{2}), "
                     r"(?P<time>\d{1,2}:\d{2}) at (?P<location>[^.]+)")),
    ("structured", R(r"^reminder: (?P<title>.+?) happens on "
                     r"(?P<date>\d{4}-\d{2}-\d{2}) at "
                     r"(?P<time>\d{1,2}:\d{2}) in (?P<location>[^.]+)")),
    ("structured", R(r"^the (?P<title>.+?) is scheduled for "
                     r"(?P<date>\d{4}-\d{2}-\d{2}) at "
                     r"(?P<time>\d{1,2}:\d{2}) in (?P<location>[^.]+)")),

    # Scheduling intent is real; the timing is not resolvable.
    ("vague",      R(r"^let us meet (?P<hint>sometime next week)")),
    ("vague",      R(r"^the (?P<title>review) could be "
                     r"(?P<hint>Friday afternoon)")),
]

ALL_FRAMES = ([("task", s, p) for s, p in TASK_FRAMES] +
              [("event", s, p) for s, p in EVENT_FRAMES])

print(f"event frames : {len(EVENT_FRAMES)}")
print(f"total frames : {len(ALL_FRAMES)}")

In [ ]:
def extract_item(core_text, message_id, sent_ts):
    """Extract one task or event from a message, or None.

    Input : core_text, message_id, and the message's own timestamp
            (needed only for the date_in_past flag).
    Output: dict of slots, or None if no frame matched.

    Rules enforced here:
      - deadline is a LITERAL date or UNRESOLVED or None. Vague
        phrases are never converted into a date.
      - time comes only from an explicit HH:MM span.
      - person is only set when a name is actually written.
      - date_in_past reports a dataset property. The date is
        preserved exactly as supplied and never corrected.
    """
    for kind, strength, pattern in ALL_FRAMES:
        m = pattern.search(core_text)
        if not m:
            continue

        g = m.groupdict()
        date = g.get("date")
        hint = g.get("hint")

        # value / referenced-but-unresolved / never-mentioned
        if date:
            deadline = date
        elif hint:
            deadline = UNRESOLVED
        else:
            deadline = None

        title = g.get("title")
        if not title and "let us meet" in core_text.lower():
            title = "Meet"
        if not title and g.get("person"):
            title = f"Call {g['person']}"
        if title:
            title = title[0].upper() + title[1:]

        # Person only when explicitly named in the text.
        person = g.get("person")
        if not person and re.search(r"\bMaya\b", core_text):
            person = "Maya"

        # Priority is a documented heuristic, not a message field.
        if date:
            priority = PRIORITY_HIGH
        elif strength == "partial":
            priority = PRIORITY_MEDIUM
        else:
            priority = PRIORITY_LOW

        date_in_past = None
        if date:
            date_in_past = bool(
                pd.Timestamp(date) < sent_ts.normalize()
            )

        return {
            "type": kind,
            "title": title,
            "description": core_text,
            "deadline": deadline,
            "date_hint": hint,
            "time": g.get("time"),
            "person": person,
            "location": g.get("location"),
            "priority": priority,
            "date_in_past": date_in_past,
            "evidence": strength,
            "source_message_id": message_id,
        }
    return None


print("extract_item() defined")
print(f"  UNRESOLVED sentinel : {UNRESOLVED!r}")
print("  None                : slot never referenced")

In [ ]:
# Extraction is attempted only for the two categories that can
# contain a task or an event. Sensitive messages are excluded so
# no credential text is copied into a description field.
_eligible = work_df["category"].isin(
    [CATEGORY_ACTION, CATEGORY_MEETING]
)

_items = []
for row in work_df[_eligible].itertuples():
    item = extract_item(row.core_text, row.message_id, row.parsed_ts)
    if item:
        _items.append(item)

for i, item in enumerate(_items, start=1):
    item["item_id"] = f"{'TASK' if item['type']=='task' else 'EVENT'}_{i:04d}"

items_df = pd.DataFrame(_items)

print(f"Eligible messages : {int(_eligible.sum())}")
print(f"Items extracted   : {len(items_df)}")
print(f"Unmatched         : {int(_eligible.sum()) - len(items_df)}")

print("\nBy type:")
for t, n in items_df["type"].value_counts().items():
    print(f"  {n:>4}  {t}")

print("\nBy evidence strength:")
for s, n in items_df["evidence"].value_counts().items():
    print(f"  {n:>4}  {s}")

_explicit = int((items_df["deadline"].notna() &
                 (items_df["deadline"] != UNRESOLVED)).sum())
print("\nDeadline slots:")
print(f"  {_explicit:>4}  explicit date")
print(f"  {int((items_df['deadline'] == UNRESOLVED).sum()):>4}  unresolved")
print(f"  {int(items_df['deadline'].isna().sum()):>4}  not referenced (None)")

print("\nOther slots (null = honestly absent):")
print(f"  time null     : {int(items_df['time'].isna().sum())}")
print(f"  person set    : {int(items_df['person'].notna().sum())}")
print(f"  location set  : {int(items_df['location'].notna().sum())}")
print(f"  date_in_past  : {int((items_df['date_in_past'] == True).sum())}")

In [ ]:
# ============================================================
# SECTION 9 - COMPLETE PROCESSING PIPELINE
# ============================================================
# One entry point that runs the whole system end to end.
# The Streamlit app in a later step imports/reuses THIS
# function - it never reimplements the logic.
# ============================================================

def process_messages(raw_df: pd.DataFrame):
    """Run the full pipeline over a raw messages dataframe.

    Input : dataframe with message_id, timestamp, sender, message
    Output: (processed_df, items_df)
              processed_df - one row per message, with category,
                             confidence, reason, sensitivity and
                             masked_text
              items_df     - one row per extracted task/event

    Stages, in order:
      1. parse timestamps and sort chronologically
      2. strip noise prefixes -> core_text, match_text
      3. detect sensitive VALUES on the ORIGINAL message
      4. classify via the precedence ladder
      5. extract tasks/events from eligible categories

    Order matters: sensitive detection runs before classification
    because the Sensitive category sits at the top of the ladder,
    and it reads the original text because match_text is
    lowercased.
    """
    df = raw_df.copy()

    # --- stage 1: chronological order -----------------------
    df["source_order"] = range(len(df))
    df["parsed_ts"] = pd.to_datetime(
        df["timestamp"], format=TIMESTAMP_FORMAT, errors="coerce"
    )
    df = df.sort_values(
        ["parsed_ts", "message_id"], kind="stable"
    ).reset_index(drop=True)

    # --- stage 2: minimal preprocessing ---------------------
    _s = df["message"].map(strip_noise_prefixes)
    df["core_text"] = [x[0] for x in _s]
    df["prefixes_removed"] = [x[1] for x in _s]
    df["match_text"] = df["core_text"].map(build_match_text)

    # --- stage 3: sensitive detection (original text) -------
    _sens = df["message"].map(detect_sensitive)
    df["is_sensitive"]       = [r["is_sensitive"] for r in _sens]
    df["is_reference_only"]  = [r["is_reference"] for r in _sens]
    df["sensitivity_types"]  = [r["types"] for r in _sens]
    df["risk_level"]         = [r["risk"] for r in _sens]
    df["recommended_action"] = [r["recommended_action"]
                                for r in _sens]
    df["masked_text"]        = [r["masked_text"] for r in _sens]
    df["sensitive_reason"]   = [r["reason"] for r in _sens]

    # --- stage 4: classification ----------------------------
    _cls = [
        classify_message(c, s)
        for c, s in zip(df["core_text"], df["is_sensitive"])
    ]
    df["category"]     = [c["category"] for c in _cls]
    df["confidence"]   = [c["confidence"] for c in _cls]
    df["reason"]       = [c["reason"] for c in _cls]
    df["matched_rule"] = [c["matched_rule"] for c in _cls]

    # --- stage 5: task / event extraction -------------------
    eligible = df["category"].isin(
        [CATEGORY_ACTION, CATEGORY_MEETING]
    )
    items = []
    for row in df[eligible].itertuples():
        it = extract_item(row.core_text, row.message_id,
                          row.parsed_ts)
        if it:
            items.append(it)

    # IDs assigned in chronological order -> deterministic.
    for i, it in enumerate(items, start=1):
        prefix = "TASK" if it["type"] == "task" else "EVENT"
        it["item_id"] = f"{prefix}_{i:04d}"

    items = pd.DataFrame(items)
    return df, items


print("process_messages() defined")
print("  stages: order -> preprocess -> sensitive -> "
      "classify -> extract")

In [ ]:
processed_df, items_df = process_messages(messages_df)

print(f"Messages processed : {len(processed_df)}")
print(f"Items extracted    : {len(items_df)}")
print(f"Chronological      : "
      f"{processed_df['parsed_ts'].is_monotonic_increasing}")
print()

# --- equivalence: pipeline vs the manual step-by-step run ---
# If these disagree, one of the two paths has a bug.
_checks = {
    "category":   processed_df["category"].equals(work_df["category"]),
    "confidence": processed_df["confidence"].equals(work_df["confidence"]),
    "reason":     processed_df["reason"].equals(work_df["reason"]),
    "sensitive":  processed_df["is_sensitive"].equals(work_df["is_sensitive"]),
    "masked":     processed_df["masked_text"].equals(work_df["masked_text"]),
}
print("Equivalence with Sections 5-8 (all must be True):")
for name, ok in _checks.items():
    print(f"  {name:<12} {ok}")
print(f"  ALL MATCH    {all(_checks.values())}")
print()

# --- determinism: same input must give the same output ------
_p2, _i2 = process_messages(messages_df)
_det_msg  = processed_df["category"].equals(_p2["category"])
_det_item = items_df.equals(_i2)
print(f"Deterministic (messages) : {_det_msg}")
print(f"Deterministic (items)    : {_det_item}")
print()

print("Category distribution:")
for cat in CATEGORY_PRECEDENCE:
    n = int((processed_df["category"] == cat).sum())
    print(f"  {n:>4}  {CATEGORY_LABELS[cat]}")

In [ ]:
# ============================================================
# SECTION 10 - VALIDATION & TESTING
# ============================================================
# The leakage test harvests the ACTUAL sensitive values from the
# original messages, then scans every artifact we intend to emit.
# Values are held in memory only, never printed or written.
# ============================================================

def harvest_sensitive_values(messages) -> set:
    """Collect the raw sensitive values present in the corpus.

    Used ONLY as the needle list for the leakage scan. The return
    value must never be printed, logged or serialised.
    """
    values = set()
    for msg in messages:
        for _stype, pattern in SENSITIVE_PATTERNS:
            for m in pattern.finditer(msg):
                v = (m.group(1) or "").strip()
                if len(v) >= 4:          # ignore trivial fragments
                    values.add(v)
    return values


def scan_for_leaks(blob: str, needles: set) -> int:
    """Return how many raw sensitive values appear in `blob`."""
    return sum(1 for v in needles if v in blob)


SENSITIVE_VALUES = harvest_sensitive_values(processed_df["message"])
print(f"Raw sensitive values harvested : {len(SENSITIVE_VALUES)}")
print("(held in memory for scanning only - never printed)\n")

# --- Self-test: the scanner must be able to FAIL -------------
# A leakage test that cannot detect a leak proves nothing.
_probe = sorted(SENSITIVE_VALUES)[0]
_poisoned = f'{{"masked_text": "Your code is {_probe}."}}'
_clean = '{"masked_text": "Your code is ******."}'

print("Scanner self-test:")
print(f"  clean sample    -> {scan_for_leaks(_clean, SENSITIVE_VALUES)} leaks (expect 0)")
print(f"  poisoned sample -> {scan_for_leaks(_poisoned, SENSITIVE_VALUES)} leaks (expect >0)")
print("  scanner can detect a leak :",
      scan_for_leaks(_poisoned, SENSITIVE_VALUES) > 0)

In [ ]:
# Columns that are SAFE to publish. Note `message` is excluded:
# raw supplied text never leaves the local environment.
SAFE_MESSAGE_COLUMNS = [
    "message_id", "timestamp", "sender", "category", "confidence",
    "reason", "is_sensitive", "is_reference_only", "risk_level",
    "recommended_action", "masked_text",
]

SAFE_ITEM_COLUMNS = [
    "item_id", "type", "title", "description", "deadline",
    "date_hint", "time", "person", "location", "priority",
    "date_in_past", "evidence", "source_message_id",
]

_artifacts = {
    "classifications": processed_df[SAFE_MESSAGE_COLUMNS].to_json(orient="records"),
    "items":           items_df[SAFE_ITEM_COLUMNS].to_json(orient="records"),
    "masked_text_col": "\n".join(processed_df["masked_text"]),
    "reason_col":      "\n".join(processed_df["reason"]),
    "title_col":       "\n".join(items_df["title"].dropna()),
    "description_col": "\n".join(items_df["description"].dropna()),
}

print("Leakage scan of output artifacts:\n")
_total_leaks = 0
for name, blob in _artifacts.items():
    n = scan_for_leaks(blob, SENSITIVE_VALUES)
    _total_leaks += n
    print(f"  {'PASS' if n == 0 else 'FAIL'}  {name:<18} leaks={n}")

print(f"\nTOTAL LEAKS: {_total_leaks}")
print("LEAKAGE TEST:", "PASS" if _total_leaks == 0 else "FAIL")

# Every sensitive row must actually differ from its original.
_unmasked = processed_df[
    processed_df["is_sensitive"] &
    (processed_df["masked_text"] == processed_df["message"])
]
print(f"\nSensitive rows unchanged by masking: {len(_unmasked)}")
if len(_unmasked):
    print("  affected IDs:", list(_unmasked["message_id"]))

In [ ]:
CHECKS = []

def record(name, passed, detail=""):
    CHECKS.append((name, bool(passed), detail))

# --- coverage ------------------------------------------------
record("all 900 messages processed",
       len(processed_df) == 900, f"{len(processed_df)}")
record("no message IDs lost",
       set(processed_df["message_id"]) == set(messages_df["message_id"]))
record("message IDs unique",
       processed_df["message_id"].is_unique)

# --- chronology ----------------------------------------------
record("processed chronologically",
       processed_df["parsed_ts"].is_monotonic_increasing)

# --- classification ------------------------------------------
record("every message has a category",
       processed_df["category"].notna().all())
record("categories all valid",
       set(processed_df["category"]).issubset(set(CATEGORY_PRECEDENCE)))
record("all six categories present",
       set(processed_df["category"]) == set(CATEGORY_PRECEDENCE))
record("every message has a reason",
       (processed_df["reason"].str.len() > 0).all())
record("confidence within declared bands",
       processed_df["confidence"].isin(CONFIDENCE.values()).all())

# --- reason integrity ----------------------------------------
# Fallback confidence must only ever appear with General Information.
_fb = processed_df[processed_df["confidence"] == CONFIDENCE["fallback"]]
record("fallback confidence only on general_information",
       (_fb["category"] == CATEGORY_GENERAL).all(), f"{len(_fb)} rows")

# --- extraction schema ---------------------------------------
record("every item has a source_message_id",
       items_df["source_message_id"].notna().all())
record("source IDs exist in the dataset",
       set(items_df["source_message_id"]).issubset(set(processed_df["message_id"])))
record("item IDs unique",
       items_df["item_id"].is_unique)
record("item types valid",
       set(items_df["type"]).issubset({"task", "event"}))
record("no item extracted from a sensitive message",
       not set(items_df["source_message_id"]) &
           set(processed_df.loc[processed_df["is_sensitive"], "message_id"]))

# --- no fabrication ------------------------------------------
_explicit = items_df["deadline"].notna() & (items_df["deadline"] != UNRESOLVED)
_iso = items_df.loc[_explicit, "deadline"].str.match(r"^\d{4}-\d{2}-\d{2}$")
record("every explicit deadline is a literal ISO date", _iso.all())
record("every explicit deadline appears in its source message",
       all(
           d in processed_df.loc[
               processed_df["message_id"] == s, "message"].iloc[0]
           for d, s in zip(items_df.loc[_explicit, "deadline"],
                           items_df.loc[_explicit, "source_message_id"])
       ))
record("every extracted time appears in its source message",
       all(
           t in processed_df.loc[
               processed_df["message_id"] == s, "message"].iloc[0]
           for t, s in zip(items_df["time"].dropna(),
                           items_df.loc[items_df["time"].notna(),
                                        "source_message_id"])
       ))
record("every named person appears in its source message",
       all(
           p in processed_df.loc[
               processed_df["message_id"] == s, "message"].iloc[0]
           for p, s in zip(items_df["person"].dropna(),
                           items_df.loc[items_df["person"].notna(),
                                        "source_message_id"])
       ))

# --- mandatory IDs -------------------------------------------
record("all 15 mandatory IDs processed",
       set(MANDATORY_IDS).issubset(set(processed_df["message_id"])))

# --- safety ---------------------------------------------------
record("no raw sensitive value in any artifact", _total_leaks == 0)

# --- report ---------------------------------------------------
print("=" * 62)
print("VALIDATION SUITE")
print("=" * 62)
for name, passed, detail in CHECKS:
    tag = "PASS" if passed else "FAIL"
    print(f"  {tag}  {name}{('  [' + detail + ']') if detail else ''}")

_n_pass = sum(1 for _, p, _ in CHECKS if p)
print("-" * 62)
print(f"  {_n_pass}/{len(CHECKS)} checks passed")
print("  RESULT:", "ALL PASS" if _n_pass == len(CHECKS) else "FAILURES PRESENT")
print("=" * 62)

In [ ]:
# ============================================================
# SECTION 11 - OUTPUT GENERATION
# ============================================================
# Schemas follow the assessment PDF examples exactly.
# The raw `message` column is NEVER written to any file.
# ============================================================

def to_records(frame: pd.DataFrame) -> list:
    """Convert a dataframe to JSON records with true nulls.

    astype(object).where(notna) keeps None as JSON null instead
    of letting it become NaN or the string "None". This matters:
    the assessment requires unresolved slots to be null.
    """
    clean = frame.astype(object).where(pd.notna(frame), None)
    return json.loads(clean.to_json(orient="records"))


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return path


# --- 1. classifications -------------------------------------
# Exact PDF schema: message_id, category, confidence, reason.
classifications = to_records(
    processed_df[["message_id", "category", "confidence", "reason"]]
)

write_json(OUTPUT_DIR / "classifications.json", classifications)
processed_df[["message_id", "category", "confidence", "reason"]].to_csv(
    OUTPUT_DIR / "classifications.csv", index=False, encoding="utf-8"
)

print(f"classifications.json : {len(classifications)} records")
print(f"classifications.csv  : written")
print("\nsample record:")
print(json.dumps(classifications[0], indent=2))

In [ ]:
# --- 2. tasks and events ------------------------------------
# PDF fields first, then the extra fields we justified:
# description, date_hint, location, date_in_past, evidence.
ITEM_FIELDS = [
    "item_id", "type", "title", "deadline", "time", "person",
    "priority", "source_message_id",
    "description", "date_hint", "location", "date_in_past",
    "evidence",
]

tasks_events = to_records(items_df[ITEM_FIELDS])
write_json(OUTPUT_DIR / "tasks_events.json", tasks_events)

print(f"tasks_events.json : {len(tasks_events)} records")
print("\nsample task:")
print(json.dumps(
    next(r for r in tasks_events if r["type"] == "task"), indent=2))
print("\nsample unresolved item:")
print(json.dumps(
    next(r for r in tasks_events if r["deadline"] == UNRESOLVED),
    indent=2))

In [ ]:
# --- 3. sensitive report ------------------------------------
# Exact PDF schema: message_id, sensitivity_type, risk,
# masked_text, recommended_action.
_sens_rows = processed_df[processed_df["is_sensitive"]].copy()
_sens_rows["sensitivity_type"] = _sens_rows["sensitivity_types"].map(
    lambda t: ", ".join(t)
)
_sens_rows = _sens_rows.rename(columns={"risk_level": "risk"})

sensitive_report = to_records(
    _sens_rows[["message_id", "sensitivity_type", "risk",
                "masked_text", "recommended_action"]]
)
write_json(OUTPUT_DIR / "sensitive_report.json", sensitive_report)

print(f"sensitive_report.json : {len(sensitive_report)} records")
print("\nsample record (value already masked):")
print(json.dumps(sensitive_report[0], indent=2))

In [ ]:
# --- 4. run summary -----------------------------------------
# Reproducibility record + honest description of what the
# numbers mean. No accuracy metric appears here, because no
# ground truth exists.
run_summary = {
    "dataset": {
        "messages_processed": int(len(processed_df)),
        "timestamp_range": [
            str(processed_df["parsed_ts"].min()),
            str(processed_df["parsed_ts"].max()),
        ],
        "processed_chronologically": bool(
            processed_df["parsed_ts"].is_monotonic_increasing),
    },
    "classification": {
        "method": "deterministic frame-matching precedence ladder",
        "counts": {
            c: int((processed_df["category"] == c).sum())
            for c in CATEGORY_PRECEDENCE
        },
        "confidence_distribution": {
            str(k): int(v) for k, v in
            processed_df["confidence"].value_counts().sort_index().items()
        },
        "confidence_note": (
            "Rule-strength scores, NOT model probabilities. "
            "No classifier was trained and no ground-truth "
            "labels exist, so no accuracy metric is reported."
        ),
    },
    "extraction": {
        "items": int(len(items_df)),
        "tasks": int((items_df["type"] == "task").sum()),
        "events": int((items_df["type"] == "event").sum()),
        "deadline_explicit": int(
            (items_df["deadline"].notna() &
             (items_df["deadline"] != UNRESOLVED)).sum()),
        "deadline_unresolved": int(
            (items_df["deadline"] == UNRESOLVED).sum()),
        "deadline_not_referenced": int(
            items_df["deadline"].isna().sum()),
        "date_in_past": int((items_df["date_in_past"] == True).sum()),
        "date_in_past_note": (
            "Dataset property: these deadlines precede their own "
            "message timestamp. Preserved literally, not corrected."
        ),
    },
    "sensitive": {
        "detected": int(processed_df["is_sensitive"].sum()),
        "reference_only_not_flagged": int(
            processed_df["is_reference_only"].sum()),
        "by_type": {
            k: int(v) for k, v in
            Counter(t for ts in processed_df["sensitivity_types"]
                    for t in ts).items()
        },
        "leakage_test_leaks": int(_total_leaks),
    },
    "validation": {
        "checks_run": len(CHECKS),
        "checks_passed": sum(1 for _, p, _ in CHECKS if p),
        "all_passed": all(p for _, p, _ in CHECKS),
    },
}

write_json(OUTPUT_DIR / "run_summary.json", run_summary)
print(json.dumps(run_summary, indent=2))

In [ ]:
# Scanning in-memory data proved the DATA was clean.
# This proves the FILES on disk are clean.

_written = sorted(OUTPUT_DIR.glob("*.json")) + \
           sorted(OUTPUT_DIR.glob("*.csv"))

print("Post-write leakage scan (reading files back from disk):\n")
_disk_leaks = 0
for path in _written:
    text = path.read_text(encoding="utf-8")
    n = scan_for_leaks(text, SENSITIVE_VALUES)
    _disk_leaks += n
    size_kb = path.stat().st_size / 1024
    tag = "PASS" if n == 0 else "FAIL"
    print(f"  {tag}  {path.name:<24} {size_kb:>7.1f} KB  leaks={n}")

print(f"\nTOTAL LEAKS ON DISK: {_disk_leaks}")
print("FILE SAFETY:", "PASS" if _disk_leaks == 0 else "FAIL - DO NOT COMMIT")

# The raw message column must not appear in any written file.
_raw_present = any(
    "message\"" in (p.read_text(encoding="utf-8")[:2000])
    for p in OUTPUT_DIR.glob("*.json")
)
print(f"Raw 'message' field in outputs: {_raw_present} (expect False)")
print(f"\nFiles written to: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# SECTION 12 - MANDATORY DEMONSTRATION CASES
# ============================================================
# Results are READ FROM the pipeline output. There is no lookup
# table and no per-ID branch anywhere in the codebase.
# Only masked_text is displayed - safe for screen recording.
# ============================================================

def show_message_result(message_id: str, width: int = 66):
    """Print the full pipeline result for one message ID.

    Reads from processed_df / items_df. Displays masked_text
    only, never the raw message column.
    """
    rows = processed_df[processed_df["message_id"] == message_id]
    if rows.empty:
        print(f"{message_id}: NOT FOUND")
        return
    r = rows.iloc[0]

    print("=" * width)
    print(f"{r['message_id']}   {r['timestamp']}   from {r['sender']}")
    print("-" * width)
    print(f"  text      : {r['masked_text']}")
    print(f"  category  : {CATEGORY_LABELS[r['category']]}")
    print(f"  confidence: {r['confidence']:.2f}"
          f"  ({CONFIDENCE_MEANING[r['confidence']]})")
    print(f"  reason    : {r['reason']}")

    if r["is_sensitive"]:
        print("-" * width)
        print(f"  SENSITIVE : {', '.join(r['sensitivity_types'])}")
        print(f"  risk      : {r['risk_level']}")
        print(f"  action    : {r['recommended_action']}")
    elif r["is_reference_only"]:
        print("-" * width)
        print("  NOTE      : references credential information "
              "but carries no value -> not flagged sensitive")

    items = items_df[items_df["source_message_id"] == message_id]
    for it in items.itertuples():
        print("-" * width)
        print(f"  EXTRACTED {it.item_id} ({it.type}, "
              f"evidence={it.evidence})")
        print(f"    title    : {it.title}")
        print(f"    deadline : {it.deadline}"
              + (f"   [hint: {it.date_hint}]"
                 if pd.notna(it.date_hint) else ""))
        print(f"    time     : {it.time}")
        print(f"    person   : {it.person}")
        print(f"    location : {it.location}")
        print(f"    priority : {it.priority}")
        if it.date_in_past is True:
            print("    NOTE     : deadline precedes the message "
                  "timestamp (dataset property, preserved)")
    print("=" * width)


print("show_message_result() defined")

In [ ]:
# Displayed in the order supplied in mandatory_demo_ids.csv.
print(f"MANDATORY DEMONSTRATION IDS ({len(MANDATORY_IDS)})\n")

for mid in MANDATORY_IDS:
    show_message_result(mid)
    print()

# Coverage summary for the video.
_demo = processed_df[processed_df["message_id"].isin(MANDATORY_IDS)]
print("Category coverage across the mandatory set:")
for cat in CATEGORY_PRECEDENCE:
    n = int((_demo["category"] == cat).sum())
    print(f"  {n:>2}  {CATEGORY_LABELS[cat]}")
print(f"\nCategories represented: {_demo['category'].nunique()} of 6")
print(f"Items extracted from mandatory messages: "
      f"{len(items_df[items_df['source_message_id'].isin(MANDATORY_IDS)])}")

In [ ]:
import inspect

# Every function that influences a message's outcome.
_pipeline_fns = [
    strip_noise_prefixes, build_match_text, detect_sensitive,
    classify_message, extract_item, process_messages,
]

_sources = "\n".join(inspect.getsource(f) for f in _pipeline_fns)

_hits = [m for m in MANDATORY_IDS if m in _sources]
_generic = re.findall(r"MSG_\d+", _sources)

print("No-hardcoding proof:")
print(f"  functions inspected      : {len(_pipeline_fns)}")
print(f"  mandatory IDs in source  : {len(_hits)}  (must be 0)")
print(f"  any MSG_ literal in source: {len(_generic)}  (must be 0)")
print(f"  RESULT: "
      f"{'PASS - no per-ID logic exists' if not _hits and not _generic else 'FAIL'}")
print()

# The rules that fired for the mandatory set are the same rules
# used across the whole corpus - not bespoke branches.
_demo_rules = set(_demo["matched_rule"])
print(f"Distinct rules fired on the mandatory 15 : {len(_demo_rules)}")
print("Each of those rules also fires elsewhere in the corpus:")
for rule in sorted(_demo_rules):
    total = int((processed_df["matched_rule"] == rule).sum())
    on_demo = int((_demo["matched_rule"] == rule).sum())
    print(f"  {on_demo:>2} of {total:>4} total   {rule[:44]}")

In [ ]:
import sys
from pathlib import Path

print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"pipeline.py exists: {(PROJECT_ROOT / 'pipeline.py').exists()}")
print(f"cwd               : {Path.cwd()}")
print()

# Make the project root importable regardless of how Jupyter set cwd.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added to sys.path : {PROJECT_ROOT}")
else:
    print("PROJECT_ROOT already on sys.path")

print("\nFiles in project root:")
for p in sorted(PROJECT_ROOT.iterdir()):
    if p.is_file():
        print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

In [ ]:
# ============================================================
# SECTION 13 - SHARED MODULE EQUIVALENCE
# ============================================================
# pipeline.py holds the single source of truth imported by both
# this notebook and app.py. Streamlit cannot import from a
# .ipynb, and duplicating the logic in app.py would create two
# classifiers that drift apart - so one flat module at project
# root is the minimal resolution.
#
# This cell proves the module reproduces the notebook exactly.
# ============================================================

import importlib
import pipeline
importlib.reload(pipeline)

mod_processed, mod_items = pipeline.process_messages(messages_df)

_eq = {
    "row count":   len(mod_processed) == len(processed_df),
    "item count":  len(mod_items) == len(items_df),
    "category":    mod_processed["category"].equals(processed_df["category"]),
    "confidence":  mod_processed["confidence"].equals(processed_df["confidence"]),
    "reason":      mod_processed["reason"].equals(processed_df["reason"]),
    "masked_text": mod_processed["masked_text"].equals(processed_df["masked_text"]),
    "is_sensitive": mod_processed["is_sensitive"].equals(processed_df["is_sensitive"]),
    "deadline":    mod_items["deadline"].equals(items_df["deadline"]),
    "title":       mod_items["title"].equals(items_df["title"]),
    "person":      mod_items["person"].equals(items_df["person"]),
}

print("pipeline.py vs notebook equivalence:\n")
for name, ok in _eq.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

print(f"\n  {sum(_eq.values())}/{len(_eq)} checks passed")
print("  RESULT:", "IDENTICAL - safe for app.py to import"
      if all(_eq.values()) else "DIVERGENT - do not proceed")

# Leakage test re-run through the module's own functions.
_v = pipeline.harvest_sensitive_values(mod_processed["message"])
_l = pipeline.scan_for_leaks("\n".join(mod_processed["masked_text"]), _v)
print(f"\n  module leakage scan: {_l} leaks (must be 0)")